# Step 3: Same-speaker real/fake dataset (Hindi + Telugu) + fair test of Model A

**Why:** In Step 2, Hindi real clips (Kathbath) and fake clips (IndicSynth) came from different sources, so a 0% error can be a shortcut.
Here every fake clip is **the same Kathbath speaker's voice**, made with **kNN-VC** voice conversion. Real and fake are then tested **clean** and through a **simulated phone line** (8 kHz, G.711 µ-law).

**Output (saved in /kaggle/working, kept when you use Save & Run All):**
- `step3_data/<lang>/<real|fake>/<speaker>/<id>.flac`: 16 kHz audio
- `step3_data/metadata.csv`: path, label, lang, speaker, gender, generator, source_speaker, split (train/dev/test by speaker)
- `step3_results.csv`: Model A EER on this fair test

**Settings:** GPU T4 x2, Internet ON, secret `HF_TOKEN`. Use **Save Version → Save & Run All (Commit)** (about 30–45 min).


In [ ]:
# CELL 1: install ONNX Runtime GPU
!pip uninstall -y -q onnxruntime
!pip install -q onnxruntime-gpu


In [ ]:
# CELL 2: setup + helpers
import os, tempfile, random
import numpy as np, pandas as pd, librosa, soundfile as sf, torch
import pyarrow.parquet as pq
from scipy.signal import butter, sosfilt, resample_poly
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, HfApi, hf_hub_download
from sklearn.metrics import roc_auc_score, roc_curve

login(token=UserSecretsClient().get_secret("HF_TOKEN"))
api = HfApi()
random.seed(0); np.random.seed(0); torch.manual_seed(0)

SR        = 16000
LANGS     = ["hindi", "telugu"]
N_SHARDS  = 6     # Kathbath train shards to download per language (more shards = more speakers)
N_SPK     = 20    # target speakers per language
N_REF     = 40    # reference clips per speaker for kNN-VC (~2.5-3 min of voice)
N_PER     = 25    # real clips AND fake clips per speaker
OUT       = "/kaggle/working/step3_data"
os.makedirs(OUT, exist_ok=True)

def decode(audio_bytes, name_hint=""):
    ext = os.path.splitext(name_hint or "")[1] or ".wav"
    with tempfile.NamedTemporaryFile(suffix=ext, delete=False) as f:
        f.write(audio_bytes)
    try:
        wav, _ = librosa.load(f.name, sr=SR, mono=True)
    finally:
        os.remove(f.name)
    return wav.astype(np.float32)

def trim(wav):
    t, _ = librosa.effects.trim(wav, top_db=30)
    return t if len(t) > SR // 2 else wav

def phone(wav):
    """Simulated phone line: 300-3400 Hz band, 8 kHz, G.711 mu-law 8-bit, back to 16 kHz."""
    x = sosfilt(butter(4, [300, 3400], btype="band", fs=SR, output="sos"), wav)
    x = resample_poly(x, 1, 2)
    x = x / (np.abs(x).max() + 1e-9) * 0.9
    mu = 255.0
    y = np.round(np.sign(x) * np.log1p(mu * np.abs(x)) / np.log1p(mu) * 127) / 127
    x = np.sign(y) * ((1 + mu) ** np.abs(y) - 1) / mu
    return resample_poly(x, 2, 1).astype(np.float32)

def eer_auc(labels, p_fake):
    labels, p_fake = np.asarray(labels), np.asarray(p_fake)
    fpr, tpr, thr = roc_curve(labels, p_fake)
    fnr = 1 - tpr
    i = np.nanargmin(np.abs(fnr - fpr))
    return {"EER_%": 100 * (fpr[i] + fnr[i]) / 2,
            "AUC": roc_auc_score(labels, p_fake),
            "threshold_at_EER": thr[i],
            "accuracy_at_0.5_%": 100 * ((p_fake >= 0.5).astype(int) == labels).mean()}

def kathbath(lang, n_shards=N_SHARDS):
    """Download evenly spaced train shards so we get many different speakers."""
    files = sorted(f.path for f in api.list_repo_tree("ai4bharat/Kathbath", path_in_repo=lang,
                                                      repo_type="dataset", recursive=True)
                   if f.path.endswith(".parquet") and "train" in os.path.basename(f.path))
    pick = sorted({files[int(i)] for i in np.linspace(0, len(files) - 1, min(n_shards, len(files)))})
    print(f"{lang}: {len(files)} train shards, downloading {len(pick)}")
    dfs = [pq.read_table(hf_hub_download("ai4bharat/Kathbath", p, repo_type="dataset")).to_pandas()
           for p in pick]
    return pd.concat(dfs, ignore_index=True)

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - check settings")


In [ ]:
# CELL 3: load kNN-VC (WavLM-Large + HiFi-GAN vocoder)
knn_vc = torch.hub.load("bshall/knn-vc", "knn_vc", prematched=True, trust_repo=True,
                        pretrained=True, device="cuda")
print("kNN-VC ready")


In [ ]:
# CELL 4: build the same-speaker dataset
rows = []
for lang in LANGS:
    df = kathbath(lang)
    counts = df.speaker_id.value_counts()
    ok = counts[counts >= N_REF + N_PER].index.tolist()
    gender = df.drop_duplicates("speaker_id").set_index("speaker_id")["gender"].astype(str).str.lower()
    fem = [s for s in ok if gender[s].startswith("f")]
    mal = [s for s in ok if not gender[s].startswith("f")]
    random.shuffle(fem); random.shuffle(mal)
    targets = fem[:N_SPK // 2] + mal[:N_SPK // 2]
    targets += [s for s in ok if s not in targets][:N_SPK - len(targets)]
    print(f"{lang}: {df.speaker_id.nunique()} speakers in shards, {len(ok)} with enough clips, using {len(targets)}")

    # speaker-disjoint split: 70% train, 15% dev, 15% test
    order = targets[:]; random.shuffle(order)
    n_tr, n_dev = int(0.7 * len(order)), int(0.15 * len(order))
    split = {s: ("train" if i < n_tr else "dev" if i < n_tr + n_dev else "test") for i, s in enumerate(order)}

    for k, spk in enumerate(targets):
        mine = df[df.speaker_id == spk].sample(frac=1, random_state=k)
        ref_rows, real_rows = mine.iloc[:N_REF], mine.iloc[N_REF:N_REF + N_PER]
        src_rows = df[df.speaker_id != spk].sample(N_PER, random_state=k)

        refs = [torch.from_numpy(trim(decode(a["bytes"], a.get("path")))) for a in ref_rows["audio_filepath"]]
        matching_set = knn_vc.get_matching_set(refs, vad_trigger_level=0)

        sdir = str(spk).replace("/", "_")
        for j, (_, r) in enumerate(real_rows.iterrows()):
            wav = trim(decode(r["audio_filepath"]["bytes"], r["audio_filepath"].get("path")))
            p = f"{OUT}/{lang}/real/{sdir}/{j:03d}.flac"
            os.makedirs(os.path.dirname(p), exist_ok=True); sf.write(p, wav, SR)
            rows.append(dict(path=p, label=0, lang=lang, speaker=spk, gender=gender[spk],
                             generator="bonafide", source_speaker=spk, split=split[spk], text=r["text"]))
        for j, (_, r) in enumerate(src_rows.iterrows()):
            src = torch.from_numpy(trim(decode(r["audio_filepath"]["bytes"], r["audio_filepath"].get("path"))))
            with torch.inference_mode():
                out = knn_vc.match(knn_vc.get_features(src), matching_set, topk=4)
            wav = trim(out.detach().cpu().numpy().astype(np.float32))
            p = f"{OUT}/{lang}/fake/{sdir}/{j:03d}.flac"
            os.makedirs(os.path.dirname(p), exist_ok=True); sf.write(p, wav, SR)
            rows.append(dict(path=p, label=1, lang=lang, speaker=spk, gender=gender[spk],
                             generator="knnvc", source_speaker=r["speaker_id"], split=split[spk], text=r["text"]))
        print(f"  [{lang}] speaker {k + 1}/{len(targets)} ({split[spk]}) done")
    del df

meta = pd.DataFrame(rows)
meta.to_csv(f"{OUT}/metadata.csv", index=False)
print(meta.groupby(["lang", "split", "label"]).size())


In [ ]:
# CELL 5: Model A (AntiDeepfake ONNX) on the fair test: clean vs phone line
import onnxruntime as ort
if hasattr(ort, "preload_dlls"):
    ort.preload_dlls()
onnx_path = hf_hub_download("SpeechAntiSpoofingBenchmarks/Wav2Vec2-Large-AntiDeepfake",
                            "wav2vec2-large-antideepfake.onnx")
sess = ort.InferenceSession(onnx_path, providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
inp = sess.get_inputs()[0]
T = inp.shape[1] if isinstance(inp.shape[1], int) else 64600
print("Running on:", sess.get_providers()[0], "| window:", T)

def crop(wav):
    if len(wav) < T:
        wav = np.tile(wav, T // len(wav) + 1)
    return wav[:T]

def score_A(wavs, bs=16):
    out = []
    for i in range(0, len(wavs), bs):
        x = np.stack([crop(w) for w in wavs[i:i + bs]])
        x = (x - x.mean(1, keepdims=True)) / (x.std(1, keepdims=True) + 1e-7)
        logits = sess.run(None, {inp.name: x.astype(np.float32)})[0]
        e = np.exp(logits - logits.max(1, keepdims=True))
        out.extend((e / e.sum(1, keepdims=True))[:, 0])       # index 0 = fake
    return np.array(out)

results = []
for lang in LANGS:
    m = meta[meta.lang == lang]
    wavs = [sf.read(p)[0].astype(np.float32) for p in m.path]
    y = m.label.values
    for channel, ws in [("clean", wavs), ("phone (8 kHz mu-law)", [phone(w) for w in wavs])]:
        r = {"model": "A: AntiDeepfake W2V-Large", "test set": f"{lang} same-speaker kNN-VC",
             "channel": channel, "n_real": int((y == 0).sum()), "n_fake": int((y == 1).sum()),
             **eer_auc(y, score_A(ws))}
        results.append(r); print(r)

res = pd.DataFrame(results).round(3)
res.to_csv("/kaggle/working/step3_results.csv", index=False)
print(res.to_string())


In [ ]:
# CELL 6: listen to a few examples (real vs fake of the SAME speaker)
from IPython.display import Audio as Play, display
for lang in LANGS:
    m = meta[meta.lang == lang]
    spk = m.speaker.iloc[0]
    for lab, name in [(0, "REAL"), (1, "FAKE (kNN-VC)")]:
        p = m[(m.speaker == spk) & (m.label == lab)].path.iloc[0]
        print(lang, name, p); display(Play(p))
